In [3]:
import pandas as pd
import requests


def fetch_open_meteo_data(
    latitude: float,
    longitude: float,
    start_date: str,
    end_date: str,
    city_name: str,
) -> pd.DataFrame:
  """Fetches historical hourly weather data for a given city from Open-Meteo API."""
  url = "https://archive-api.open-meteo.com/v1/archive"
  params = {
      "latitude": latitude,
      "longitude": longitude,
      "start_date": start_date,
      "end_date": end_date,
      "hourly": [
          "temperature_2m",
          "relative_humidity_2m",
          "precipitation",
          "wind_speed_10m",
      ],
      "timezone": "Europe/Lisbon",
  }

  response = requests.get(url, params=params)
  response.raise_for_status()
  data = response.json()

  # Parse hourly response into a Pandas DataFrame
  df = pd.DataFrame(data["hourly"])
  df["time"] = pd.to_datetime(df["time"])
  df["city"] = city_name
  df.rename(
      columns={
          "temperature_2m": "temperature",
          "relative_humidity_2m": "humidity",
          "precipitation": "precipitation",
          "wind_speed_10m": "wind_speed",
      },
      inplace=True,
  )

  # Ensure chronological order
  df.sort_values(by="time", inplace=True)
  df.reset_index(drop=True, inplace=True)
  return df


# Define parameters (check your Moodle/Assignment details for exact historical dates)
START_DATE = "2023-01-01"
END_DATE = "2023-12-31"

df_aveiro = fetch_open_meteo_data(
    40.6405, -8.6538, START_DATE, END_DATE, "Aveiro"
)
df_viseu = fetch_open_meteo_data(40.6572, -7.9142, START_DATE, END_DATE, "Viseu")

# Save raw snapshots to your audit folder data/
df_aveiro.to_csv("../data/raw_aveiro.csv", index=False)
df_viseu.to_csv("../data/raw_viseu.csv", index=False)

In [5]:
df_viseu

,time,temperature,humidity,precipitation,wind_speed,city
0,2023-01-01 00:00:00,11.7,91,2.1,18.6,Viseu
1,2023-01-01 01:00:00,11.4,91,1.1,16.4,Viseu
2,2023-01-01 02:00:00,11.7,88,0.6,19.6,Viseu
3,2023-01-01 03:00:00,11.9,86,0.1,20.8,Viseu
4,2023-01-01 04:00:00,12.1,84,0.0,21.7,Viseu
...,...,...,...,...,...,...
8755,2023-12-31 19:00:00,7.8,77,0.0,6.8,Viseu
8756,2023-12-31 20:00:00,6.9,80,0.0,5.6,Viseu
8757,2023-12-31 21:00:00,6.0,85,0.0,7.3,Viseu
8758,2023-12-31 22:00:00,5.4,87,0.0,6.3,Viseu


In [6]:
# 1. Parse the time
df_aveiro["time"] = pd.to_datetime(df_aveiro["time"])
df_viseu["time"] = pd.to_datetime(df_viseu["time"])


# 2. Sort observations chronologically
df_aveiro = df_aveiro.sort_values("time").reset_index(drop=True)
df_viseu = df_viseu.sort_values("time").reset_index(drop=True)


# 3. Check for duplicate times
duplicates_aveiro = df_aveiro["time"].duplicated().sum()
duplicates_viseu = df_viseu["time"].duplicated().sum()

print(f"Aveiro duplicate times: {duplicates_aveiro}")
print(f"Viseu duplicate time: {duplicates_viseu}")

Aveiro duplicate times: 0
Viseu duplicate time: 0


In [7]:
# Check the number of missing values in each column
print("Missing values - Aveiro:")
print(df_aveiro.isna().sum())

print("\nMissing values - Viseu:")
print(df_viseu.isna().sum())

Missing values - Aveiro:
time             0
temperature      0
humidity         0
precipitation    0
wind_speed       0
city             0
dtype: int64

Missing values - Viseu:
time             0
temperature      0
humidity         0
precipitation    0
wind_speed       0
city             0
dtype: int64


In [11]:
def check_missing_days(df, name):
    # Extract only the calendar date from the timestamp
    dates = df["time"].dt.normalize()

    # Create every calendar day between the first and last observation
    expected_dates = pd.date_range(
        start=dates.min(),
        end=dates.max(),
        freq="D"
    )

    # Find days that are not present in the dataset
    missing_dates = expected_dates.difference(dates)

    print(f"{name}:")
    
    if len(missing_dates) == 0:
        print("No missing days.")
    else:
        print(f"Missing days: {len(missing_dates)}")
        print(missing_dates)


check_missing_days(df_aveiro, "Aveiro")
check_missing_days(df_viseu, "Viseu")

Aveiro:
No missing days.
Viseu:
No missing days.
